In [2]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report

def relu(Z):
    return np.maximum(0, Z)

def relu_derivative(Z):
    return (Z > 0).astype(float)

def softmax(Z):
    exp_Z = np.exp(Z - np.max(Z, axis=-1, keepdims=True))
    return exp_Z / np.sum(exp_Z, axis=-1, keepdims=True)

def compute_loss(Y, Y_hat):
    m = Y.shape[0]
    epsilon = 1e-15
    Y_hat = np.clip(Y_hat, epsilon, 1 - epsilon)
    return -np.sum(Y * np.log(Y_hat)) / m

df = pd.read_csv("/content/drive/MyDrive/diabetes.csv")

X = df.drop("Outcome", axis=1).values
y_raw = df["Outcome"].values
y_onehot = np.eye(2)[y_raw]

X_train, X_test, y_train, y_test = train_test_split(
    X, y_onehot, test_size=0.2, random_state=42, stratify=y_raw
)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

def forward_propagation(X, params):
    W1, b1 = params["W1"], params["b1"]
    W2, b2 = params["W2"], params["b2"]

    Z1 = np.dot(X, W1) + b1
    A1 = relu(Z1)

    Z2 = np.dot(A1, W2) + b2
    A2 = softmax(Z2)

    cache = {"X": X, "Z1": Z1, "A1": A1, "Z2": Z2, "A2": A2}
    return A2, cache

def backward_propagation(Y, cache, params):
    m = Y.shape[0]
    X, Z1, A1, Z2, A2 = cache["X"], cache["Z1"], cache["A1"], cache["Z2"], cache["A2"]
    W2 = params["W2"]

    dZ2 = A2 - Y
    dW2 = np.dot(A1.T, dZ2) / m
    db2 = np.sum(dZ2, axis=0, keepdims=True) / m

    dA1 = np.dot(dZ2, W2.T)
    dZ1 = dA1 * relu_derivative(Z1)
    dW1 = np.dot(X.T, dZ1) / m
    db1 = np.sum(dZ1, axis=0, keepdims=True) / m

    grads = {"dW1": dW1, "db1": db1, "dW2": dW2, "db2": db2}
    return grads

np.random.seed(42)
input_dim = X_train.shape[1]
hidden_dim = 32
output_dim = 2

params = {
    "W1": np.random.randn(input_dim, hidden_dim) * np.sqrt(2.0 / input_dim),
    "b1": np.zeros((1, hidden_dim)),
    "W2": np.random.randn(hidden_dim, output_dim) * np.sqrt(2.0 / hidden_dim),
    "b2": np.zeros((1, output_dim))
}

learning_rate = 0.1

y_pred_prob, cache = forward_propagation(X_train, params)
grads = backward_propagation(y_train, cache, params)

params["W1"] -= learning_rate * grads["dW1"]
params["b1"] -= learning_rate * grads["db1"]
params["W2"] -= learning_rate * grads["dW2"]
params["b2"] -= learning_rate * grads["db2"]

test_probs, _ = forward_propagation(X_test, params)
y_test_pred = np.argmax(test_probs, axis=1)
y_test_true = np.argmax(y_test, axis=1)

print("\nAccuracy =", accuracy_score(y_test_true, y_test_pred))
print("\nClassification Report:\n", classification_report(y_test_true, y_test_pred))


Accuracy = 0.6623376623376623

Classification Report:
               precision    recall  f1-score   support

           0       0.68      0.89      0.77       100
           1       0.54      0.24      0.33        54

    accuracy                           0.66       154
   macro avg       0.61      0.57      0.55       154
weighted avg       0.63      0.66      0.62       154

